# CVM Free Cash Flow Pipeline

Extracts the historical **free cash flow (FCF)**, **net debt** and **shares outstanding** of Brazilian listed companies from the
open data portal of the CVM (Comissão de Valores Mobiliários), as the first step of a
discounted cash flow (DCF) valuation.

**Pipeline overview**

1. **Sources**: list the yearly DFP zip files on the CVM website, download them (with a local cache) and read the cash flow statement.
2. **Cleaning**: keep the latest fiscal year and the most recent document of each company, drop duplicates and normalize the currency scale.
3. **Metrics**: compute operating cash flow (CFO), capex and FCF for each company and year.
4. **Registry**: add sector and listing information from the CVM company registry and flag financial and investable companies.
5. **Net debt**: compute gross debt (including leases), cash and net debt from the balance sheet.
6. **Shares**: compute shares outstanding (issued minus treasury) from the share capital file.

All monetary values are in **thousands of reais**; `SHARES` is in number of shares. Capex is negative (cash out), so `FCF = CFO + CAPEX`.

**Code structure.** The functions live in the `cvm_valuation` package; this notebook runs the pipeline and documents the decisions behind it.

| Module | Contents |
|---|---|
| `config.py` | Paths, URLs, capex terms and financial sectors |
| `sources.py` | `find_zips`, `complete_years`, `get_file`, `open_csv` |
| `accounts.py` | `find_account` |
| `cleaning.py` | `split_by_company` |
| `fcf.py` | `find_capex`, `find_cfo`, `build_fcf_table`, `build_fcf_history` |
| `registry.py` | `prepare_registry`, `add_flags` |
| `balance.py` | `find_debt`, `find_cash`, `build_balance_table`, `build_balance_history` |
| `shares.py` | `prepare_shares`, `build_shares_history` |

The valuation itself (tickers, prices and DCF) is in `02_valuation.ipynb`, which reads the table saved at the end of this notebook.

> Sections 1 and 2 only import and explain. Section 3 is the first one that downloads data.

## 1. Setup

In [1]:
import pandas as pd

from cvm_valuation.config import DFP_URL, CAD_URL, DATA_DIR
from cvm_valuation.sources import find_zips, complete_years, get_file
from cvm_valuation.fcf import build_fcf_history
from cvm_valuation.registry import prepare_registry, add_flags
from cvm_valuation.balance import build_balance_history
from cvm_valuation.shares import build_shares_history

pd.set_option("display.max_rows", None)

### 1.1 Configuration (`config.py`)

Everything that may need adjusting lives in `config.py`, in one place:

- **Paths and URLs.** `DATA_DIR` is the local cache for the downloaded zip files (ignored by Git). It is built from the location of `config.py` itself, so it points to the same `data/` folder wherever the code runs from.
- **Capex rule.** A line of the investing section (`6.02`) counts as capex if its description matches `CAPEX_TERMS` and does **not** match `EXCLUDE_TERMS`. The list describes what capex *is* (an allowlist) on purpose: an unknown line is left out instead of being counted, so errors stay small.
- **Financial sectors.** Banks, insurers and similar companies are flagged, since free cash flow is not a meaningful measure for them.

The terms are regular expressions: `s?` makes the `s` optional and `.` matches any character (useful for accents).

## 2. How the pipeline works

### 2.1 Data sources (`sources.py`)

- `find_zips` reads the CVM directory page and maps each year to its zip URL (web scraping).
- `complete_years` drops years whose files are still incomplete: the current year and, before April, the previous one (companies have until the end of March to file).
- `get_file` downloads a file into memory, caching it in `DATA_DIR`. Use `use_cache=False` for files that change often, such as the company registry.
- `open_csv` reads one statement from a zip, e.g. the consolidated cash flow statement (`DFC_MI`, `con`).

### 2.2 Cleaning (`cleaning.py`)

The raw CVM files need four corrections before any calculation, all made by `split_by_company`. Each one was found in the real data:

| Problem | Example | Fix |
|---|---|---|
| Each file repeats the previous year as a comparative | Every account appears as `ÚLTIMO` and `PENÚLTIMO` | Keep `ÚLTIMO` only |
| Fully duplicated rows | 320 identical rows in 2025 (FGR Incorporações, VLI Multimodal) | `drop_duplicates` |
| Values in units instead of thousands | 7 companies in 2025 report in `UNIDADE` | Divide by 1,000 |
| More than one document per company | Estrela (8427) filed two DFPs in 2011: a 3-month one and the full year | Keep the latest `DT_REFER`, then the latest `VERSAO` |

`DT_REFER` is text, but `max()` still works because dates are in `YYYY-MM-DD` format, where alphabetical and chronological order match.

### 2.3 Cash flow metrics (`fcf.py`)

- **CFO** is the standardized account `6.01` (net cash from operating activities). `find_cfo` returns `NaN` when the account is missing and raises an error when it appears more than once, instead of silently summing duplicates. This check is what revealed the Estrela case above.
- **Capex** has no standard account: each company names its own lines inside `6.02`. `find_capex` sums the lines whose description matches the capex terms and not the exclusion terms. Beyond fixed and intangible assets, the terms cover concession contract assets (utilities), biological assets (pulp, agribusiness) and investment properties (shopping malls).

Validation: for Cemig (2453) in 2025, the rule finds R$ 6.46 billion, against R$ 6.63 billion of "investimentos realizados" in the company's investor relations spreadsheet. The difference is expected: the cash flow statement is on a cash basis and the release on an accrual basis.

### 2.4 Building the history (`fcf.py`)

`build_fcf_table` computes the metrics for every company in one year, and `build_fcf_history` repeats the whole pipeline for each year and stacks the results into one table (one row per company and year).

`YEAR` is the year of the CVM file. For companies with a non-calendar fiscal year (e.g. ending in March), it refers to the fiscal year ending in that year

## 3. Load data

This is the only section that accesses the internet. The first run downloads one zip per year (about 13 MB each) into `DATA_DIR`; later runs read them from disk. To refresh a year, delete its file from the `data/` folder.

The company registry changes often, so it is always downloaded (`use_cache=False`).

In [2]:
zip_links = find_zips(DFP_URL)
dfp_links = complete_years(zip_links)
history = build_fcf_history(dfp_links)
balance = build_balance_history(dfp_links)
shares_history = build_shares_history(dfp_links)
cad = pd.read_csv(get_file(CAD_URL, use_cache=False), sep=";", encoding="latin-1")

Processing 2010...
Processing 2011...
Processing 2012...
Processing 2013...
Processing 2014...
Processing 2015...
Processing 2016...
Processing 2017...
Processing 2018...
Processing 2019...
Processing 2020...
Processing 2021...
Processing 2022...
Processing 2023...
Processing 2024...
Processing 2025...
Processing 2010...
Processing 2011...
Processing 2012...
Processing 2013...
Processing 2014...
Processing 2015...
Processing 2016...
Processing 2017...
Processing 2018...
Processing 2019...
Processing 2020...
Processing 2021...
Processing 2022...
Processing 2023...
Processing 2024...
Processing 2025...
Processing 2010...
No share capital file for 2010, skipping
Processing 2011...
No share capital file for 2011, skipping
Processing 2012...
No share capital file for 2012, skipping
Processing 2013...
No share capital file for 2013, skipping
Processing 2014...
No share capital file for 2014, skipping
Processing 2015...
No share capital file for 2015, skipping
Processing 2016...
No share capi

## 4. Company registry (`registry.py`)

The CVM registry adds the sector and listing status of each company.

**Repeated rows.** A company traded in more than one market (e.g. `BOLSA` and `BALCÃO ORGANIZADO`) appears once per market. `prepare_registry` keeps only one row per company, and sorting `TP_MERC` in descending order makes the `BOLSA` row the one that is kept. Without this step, the merge would duplicate rows in the history.

In [3]:
registry = prepare_registry(cad)

### 4.1 Flags

`add_flags` merges the registry into the history and adds two columns. Companies are **flagged, not removed**, so the history stays complete. Removing companies that are no longer listed would keep only the survivors and make any historical comparison look better than reality (survivorship bias).

- `IS_FINANCIAL`: the sector is one of `FINANCIAL_SECTORS`, including the holding company versions ("Emp. Adm. Part. - Bancos", ...).
- `IS_INVESTABLE`: the company could be analyzed and bought **today**: active registration, category A (allowed to have traded shares), listed on the exchange, operating normally (not in judicial recovery) and not financial. Since the registry describes the present, this flag refers to today for every year of the history.

The merge uses `how="left"` so that companies missing from the registry stay in the history, with empty registry columns.

The same cell then merges the net debt (`DEBT`, `CASH`, `NET_DEBT`, from `balance.py`) and the shares outstanding (`SHARES`, from `shares.py`), using `CD_CVM` and `YEAR` as keys. Leases are counted as debt (IFRS 16). The `assert` checks that no merge duplicated rows.

In [4]:
history_full = add_flags(history, registry)
rows_before = len(history_full)
history_full = history_full.merge(balance, on=["CD_CVM", "YEAR"], how="left")
history_full = history_full.merge(shares_history, on=["CD_CVM", "YEAR"], how="left")
assert len(history_full) == rows_before, "merge duplicated rows"

## 5. Save the table

The final table is saved in `DATA_DIR`, so `02_valuation.ipynb` can use it without running the whole pipeline again. Like the zip files, it is not versioned.

In [5]:
history_full.to_csv(DATA_DIR / "history_full.csv", index=False)
history_full.shape

(6119, 18)

## 6. Known limitations

- **Capex sign.** A few companies report acquisitions with a positive sign (e.g. Even in 2025). The value may be net of sales or a filing error; these cases should be reviewed by hand.
- **Currency scale.** Some companies seem to report a wrong `ESCALA_MOEDA` (e.g. Padtec in 2025). Values should be sanity-checked against company filings.
- **Power transmission.** Part of the capex of transmission companies may be classified in operating activities, as contract assets, and is not captured by the `6.02` rule.
- **Direct method.** Only the indirect-method cash flow statement (`DFC_MI`) is used; companies that report with the direct method (`DFC_MD`) are missing.
- **IFRS 16 (2019).** Since 2019, lease payments are classified as financing, which raises the operating cash flow of companies with many leased stores. Years before and after 2019 are not fully comparable.
- **Interest paid.** Some companies report interest paid within operating activities. For a free cash flow to the firm, it should be added back.
- **Share count history.** The share capital file (`composicao_capital`) is only included in the DFP zips from 2020 onwards, so `SHARES` is empty for 2010–2019. The valuation only needs the current share count, so this does not affect it.